<h1>Collaborative Filtering based
Recommender System using K Nearest
Neighbor</h1>

<h3>Load and exploring dataset</h3>


In [35]:
import pandas as pd
import pandas as pd
from pandas import isnull
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
from lazypredict.Supervised import LazyClassifier
from lazypredict.Supervised import LazyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, SelectPercentile
from sklearn.datasets import make_classification
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
rating_url = "final_project.ods"
rating_df = pd.read_excel(rating_url)

rating_df.head()


,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


<h1>EDA</h1>

In [8]:
rating_df.head()

,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


In [9]:
rating_df.describe()

,title,location,description,function,industry,career_level
count,8074,8074,8073,8074,8074,8074
unique,6790,1081,7973,19,352,6
top,Account Manager,"New York City, NY",Practice Medicine in Northern California with ...,sales,Software Companies,senior_specialist_or_project_manager
freq,58,401,5,3151,688,4338


In [10]:
rating_df.dtypes

title           object
location        object
description     object
function        object
industry        object
career_level    object
dtype: object

<h1>Implementation Option 1: Use Surprise library </h1>

<h1>Load Data</h1>

In [13]:
def filter_location(location):
    if location[-2:].isupper() and location[-3] == " ":
        return location[-2:]
    else:
        return location

data = pd.read_excel("final_project.ods",dtype=str)
print(pd.isnull(data).sum())
data["location"]=data["location"].apply(filter_location)
data=data.dropna()
target="career_level"
x=data.drop(target,axis=1)
y=data[target]

title           0
location        0
description     1
function        0
industry        0
career_level    0
dtype: int64


<h1>Train test</h1>

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=100,stratify=y
)
print(y_train.value_counts())

career_level
senior_specialist_or_project_manager      3469
manager_team_leader                       2138
bereichsleiter                             768
director_business_unit_leader               56
specialist                                  24
managing_director_small_medium_company       3
Name: count, dtype: int64


<h1>Test và tối ưu với mô hình RandomForestClassifier</h1>
<h2>lý do: vì mô hình RandomForestClassifier nhanh và nhẹ, đạt độ acc ổn, dễ diễn giải</h2>

In [39]:
transformer = ColumnTransformer(transformers=[
    ("title",TfidfVectorizer(stop_words="english"),"title"),
    ("location,function",OneHotEncoder(handle_unknown="ignore"),["location","function"]),
    ("desc",TfidfVectorizer(ngram_range=(1,2),min_df=0.01,max_df=0.95,stop_words="english",),"description"),
    ("industry",TfidfVectorizer(stop_words="english"),"industry")
]
)
output=transformer.fit_transform(x_train)
print(output.shape)

model = Pipeline(steps=[
    ("preprocessor", transformer),
    ("select",SelectKBest(k=300)),
    # ("select",SelectPercentile(percentile=50)),
    ("classifier", RandomForestClassifier(random_state=100,criterion="gini",n_jobs=-1,n_estimators=100)),
])


(6458, 7903)


In [20]:
params = {
    "select__k": [100,200,300,500],
    "classifier__n_estimators": [50,100,200],
    "classifier__criterion": ['gini','entropy']
}
grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    scoring="recall_weighted",
    cv=6,
    verbose=3,
    n_jobs=-1
)
grid.fit(x_train, y_train)
print(grid.best_params_)
print(grid.best_score_)
best_model = grid.best_estimator_
# =>
# {'classifier__criterion': 'gini', 'classifier__n_estimators': 50, 'select__k': 100}


Fitting 6 folds for each of 24 candidates, totalling 144 fits
{'classifier__criterion': 'gini', 'classifier__n_estimators': 50, 'select__k': 100}
nan


In [21]:
model.fit(x_train, y_train)
y_predict=model.predict(x_test)
print(classification_report(y_test,y_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.74      0.13      0.22       192
         director_business_unit_leader       1.00      0.14      0.25        14
                   manager_team_leader       0.63      0.75      0.69       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.83      0.90      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.75      1615
                             macro avg       0.53      0.32      0.34      1615
                          weighted avg       0.75      0.75      0.72      1615



<h1>Test với mô hình LDA</h1>

In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
from lazypredict.Supervised import LazyClassifier
from lazypredict.Supervised import LazyRegressor
import lazypredict.Supervised
import tqdm
lazypredict.Supervised.notebook_tqdm = tqdm.tqdm

def filter_location(location):
    if location[-2:].isupper() and location[-3] == " ":
        return location[-2:]
    else:
        return location

data = pd.read_excel("final_project.ods",dtype=str)
data["location"]=data["location"].apply(filter_location)
data=data.dropna()
target="career_level"
x=data.drop(target,axis=1)
y=data[target]
print(y.value_counts())

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=100,stratify=y
)
print(y_train.value_counts())

vectorizer_des = TfidfVectorizer(ngram_range=(1,2),min_df=0.01,max_df=0.95,stop_words="english",max_features=2000)
x_des_train=vectorizer_des.fit_transform(x_train['description'])
x_des_test=vectorizer_des.transform(x_test['description'])

vectorizer_tit = TfidfVectorizer(ngram_range=(1,2),min_df=0.01,max_df=0.95,stop_words="english",max_features=1000)
x_tit_train= vectorizer_tit.fit_transform(x_train["title"])
x_tit_test=vectorizer_tit.transform(x_test["title"])

vectorizer_ind = TfidfVectorizer(ngram_range=(1,2),min_df=0.01,max_df=0.95,stop_words="english",max_features=100)
x_ind_train=vectorizer_ind.fit_transform(x_train["industry"])
x_ind_test=vectorizer_ind.transform(x_test["industry"])

encoder_loc=OneHotEncoder(sparse_output=False,handle_unknown="ignore")
x_loc_train=encoder_loc.fit_transform(x_train[["location"]])
x_loc_test=encoder_loc.transform(x_test[["location"]])

encoder_fun=OneHotEncoder(sparse_output=False,handle_unknown="ignore")
x_fun_train=encoder_fun.fit_transform(x_train[["function"]])
x_fun_test=encoder_fun.transform(x_test[["function"]])



X_train = hstack([
    x_des_train,
    x_tit_train,
    x_loc_train,
    x_fun_train,
    x_ind_train
])

X_test = hstack([
    x_des_test,
    x_tit_test,
    x_loc_test,
    x_fun_test,
    x_ind_test
])


cfs = LazyClassifier(
    verbose=0,
    ignore_warnings=True,
)
X_train_df = pd.DataFrame(X_train.toarray())
X_test_df = pd.DataFrame(X_test.toarray())

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, accuracy_score

best_model = LinearDiscriminantAnalysis()

best_model.fit(X_train_df, y_train)

y_pred = best_model.predict(X_test_df)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


career_level
senior_specialist_or_project_manager      4337
manager_team_leader                       2672
bereichsleiter                             960
director_business_unit_leader               70
specialist                                  30
managing_director_small_medium_company       4
Name: count, dtype: int64
career_level
senior_specialist_or_project_manager      3469
manager_team_leader                       2138
bereichsleiter                             768
director_business_unit_leader               56
specialist                                  24
managing_director_small_medium_company       3
Name: count, dtype: int64
Accuracy: 0.7188854489164087
                                        precision    recall  f1-score   support

                        bereichsleiter       0.42      0.36      0.39       192
         director_business_unit_leader       0.70      0.50      0.58        14
                   manager_team_leader       0.62      0.66      0.64       534
managing

<h2>Tìm tham số tối ưu</h2>

In [38]:
model = Pipeline([
    ('select', SelectKBest(score_func=f_classif)),
    ('classifier', LinearDiscriminantAnalysis())
])
params = [
    {
        "select__k": [100, 200, 300, 500],
        "classifier__solver": ['svd']
    },
    {
        "select__k": [100, 200, 300, 500],
        "classifier__solver": ['lsqr', 'eigen'],
        "classifier__shrinkage": ['auto', 0.1, 0.5, 0.9] 
    }
]
grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    scoring="recall_weighted",
    cv=6,
    verbose=3,
    n_jobs=-1
)
grid.fit(X_train_df, y_train)
print("Best Parameters:", grid.best_params_)
print("Best Recall Score:", grid.best_score_)
best_model = grid.best_estimator_

#Best Parameters: {'classifier__solver': 'svd', 'select__k': 100}

Fitting 6 folds for each of 36 candidates, totalling 216 fits
Best Parameters: {'classifier__solver': 'svd', 'select__k': 100}
Best Recall Score: nan


<h2>Áp dụng</h2>

In [43]:
best_model = Pipeline([
    ('select', SelectKBest(score_func=f_classif, k=100)),
    ('classifier', LinearDiscriminantAnalysis(solver='svd'))
])

best_model.fit(X_train_df, y_train)
y_pred = best_model.predict(X_test_df)
print(classification_report(y_test, y_pred))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.49      0.39      0.43       192
         director_business_unit_leader       0.50      0.29      0.36        14
                   manager_team_leader       0.68      0.62      0.65       534
managing_director_small_medium_company       0.25      1.00      0.40         1
  senior_specialist_or_project_manager       0.82      0.90      0.86       868
                            specialist       0.07      0.17      0.10         6

                              accuracy                           0.74      1615
                             macro avg       0.47      0.56      0.47      1615
                          weighted avg       0.73      0.74      0.73      1615



<h1>Implementation Option 2: Use numpy, pandas, and sklearn</h1>

In [30]:
import numpy as np

class ScratchLDA:
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.linear_discriminants = None
        self.classes = None
        self.projected_centroids = []

    def fit(self, X, y):
        n_features = X.shape[1]
        self.classes = np.unique(y)
        
        mean_overall = np.mean(X, axis=0)
        S_W = np.zeros((n_features, n_features))
        S_B = np.zeros((n_features, n_features))
        
        self.centroids = []

        for c in self.classes:
            X_c = X[y == c]
            mean_c = np.mean(X_c, axis=0)
            self.centroids.append(mean_c)
            S_W += (X_c - mean_c).T.dot(X_c - mean_c)
            n_c = X_c.shape[0]
            mean_diff = (mean_c - mean_overall).reshape(n_features, 1)
            S_B += n_c * (mean_diff).dot(mean_diff.T)
        A = np.linalg.pinv(S_W).dot(S_B)
        eigenvalues, eigenvectors = np.linalg.eig(A)
        eigenvectors = eigenvectors.T
        idxs = np.argsort(abs(eigenvalues))[::-1]
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]
        if self.n_components is None:
            self.n_components = len(self.classes) - 1
            
        self.linear_discriminants = eigenvectors[0:self.n_components]
        for mean_c in self.centroids:
            self.projected_centroids.append(np.dot(mean_c, self.linear_discriminants.T))

    def transform(self, X):
        return np.dot(X, self.linear_discriminants.T)
        
    def predict(self, X):
        X_projected = self.transform(X)
        y_pred = []
        for x in X_projected:
            distances = [np.linalg.norm(x - centroid) for centroid in self.projected_centroids]
            y_pred.append(self.classes[np.argmin(distances)])
            
        return np.array(y_pred)

In [46]:
num_classes = len(np.unique(y_train))
n_components = min(5, num_classes - 1)

lda_model = ScratchLDA(n_components=n_components) 
lda_model.fit(X_train_df.to_numpy(), y_train.to_numpy())
X_train_reduced = lda_model.transform(X_train_df.to_numpy())
y_pred_c = lda_model.predict(X_test_df.to_numpy())
from sklearn.metrics import accuracy_score
print("Dự đoán từ code 10 mẫu đầu:", y_pred_c[:10])
print("Độ chính xác (Accuracy):", accuracy_score(y_test, y_pred_c))

Dự đoán từ code 10 mẫu đầu: ['manager_team_leader' 'manager_team_leader'
 'senior_specialist_or_project_manager'
 'senior_specialist_or_project_manager' 'manager_team_leader'
 'senior_specialist_or_project_manager' 'bereichsleiter'
 'senior_specialist_or_project_manager' 'manager_team_leader'
 'manager_team_leader']
Độ chính xác (Accuracy): 0.713312693498452
